In [5]:
import os
import glob
import tiktoken
import hashlib
import json
from pathlib import Path
import numpy as np
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [6]:
from dotenv import load_dotenv

MODEL = "openai/gpt-oss-120b"
# MODEL = "gpt-4.1-nano"
load_dotenv(override=True)

True

In [7]:
knowledge_base_path = "../knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)

entire_knowledge_base = ""
for file in files:
    with open(file, "r", encoding="utf-8") as f:
        entire_knowledge_base += f.read() + "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base)}")

Total characters in knowledge base: 304434


In [8]:
# Tokens.
encodings = tiktoken.encoding_for_model("gpt-oss-120b")
tokens = encodings.encode(entire_knowledge_base)
print(f"Total tokens in knowledge base: {len(tokens)}")

Total tokens in knowledge base: 63555


In [9]:
# Load the knowledge base in Langchain loaders.
folders = glob.glob("../knowledge-base/*")
documents = []

for folder in folders:
    # Get the folder name of the document (eg. "company", "contracts" etc.)
    doc_type = os.path.basename(folder)
    # Use the Langchain DirectoryLoader to load all the markdown files in the folder.
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    # Return a list of Langchain Document objects, where each document is a markdown file. Add the doc_type as metadata to each document.
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 76 documents


In [10]:
# Divide into chunks using the RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")
print(f"First chunk: {chunks[0]}...")
print(f"Second chunk: {chunks[1]}...")

# Count the unique doc_types in the chunks.
chunk_doc_types = [doc.metadata.get("doc_type") for doc in chunks]
print(f"Document types in chunks: {set(chunk_doc_types)}")

Total chunks created: 413
First chunk: page_content='# Contract with EverGuard Insurance for Rellm: AI-Powered Enterprise Reinsurance Solution

**Contract Number:** IG-2023-EG  
**Effective Date:** January 1, 2024  
**Expiration Date:** December 31, 2026  

## Terms

1. **Parties**: This agreement is made between Insurellm, located at 123 Innovation Drive, Tech City, USA, and EverGuard Insurance, located at 456 Safety Lane, Protectville, USA.
   
2. **Product Description**: This contract pertains to the use of the Rellm platform, an AI-powered enterprise reinsurance solution provided by Insurellm. EverGuard Insurance will implement Rellm to enhance its reinsurance operations.

3. **Payment Terms**: EverGuard Insurance agrees to pay Insurellm a monthly fee of $10,000 for the duration of this contract, covering the Professional Plan features of Rellm, which includes all advanced integrations and priority customer support.' metadata={'source': '../knowledge-base/contracts/Contract with Ev

In [11]:
# Create an utility to check if any of the loaded documents have changed
# based on hash of the content. This will be used to update the vector database with only the changed documents.
HASH_FILE = "doc_hash.json"

def compute_dir_hash(base_dir):
    sha = hashlib.sha256()
    for path in sorted(Path(base_dir).rglob("*")):
        if path.is_file():
            sha.update(path.read_bytes())
    return sha.hexdigest()

def needs_rebuild(base_dir):
    new_hash = compute_dir_hash(base_dir)

    if not Path(HASH_FILE).exists():
        return True, new_hash

    old_hash = json.loads(Path(HASH_FILE).read_text())["hash"]
    return new_hash != old_hash, new_hash


In [12]:
# Now create the vectors and store in Chroma vector database.

os.environ["TRANSFORMERS_OFFLINE"] = "1"

embeddings = HuggingFaceEmbeddings(
    model_name=r"all-MiniLM-L6-V2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

# Now use this embedding model to create the vector database in Chroma.
PERSISTENT_DIR = "vector_db"
changed, new_hash = needs_rebuild("../knowledge-base")

if changed:
    print("Documents have changed. Rebuilding the vector database...")
    # Create the Chroma vector database and persist it to disk.
    vectordb = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=PERSISTENT_DIR
    )
    # Update the hash file with the new hash.
    Path(HASH_FILE).write_text(json.dumps({"hash": new_hash}))
    print("✅ Vector DB rebuilt")
else:
    print("Documents have not changed. Loading existing vector database...")
    vectordb = Chroma(
        persist_directory=PERSISTENT_DIR,
        embedding_function=embeddings
    )
    print("✅ Vector DB loaded from disk")

Documents have not changed. Loading existing vector database...
✅ Vector DB loaded from disk


In [13]:
# Try a semantic search to test the vector database.
query = "What are the different platforms this company has expanded into after restructuring?"
results = vectordb.similarity_search(query, k=3)
print("Top 3 results:")
for i, result in enumerate(results):
    print(f"""{i+1}:
    Page Content: {result.page_content[:200]}...
    Tag: {result.metadata['doc_type']}
    Source: {result.metadata['source']}""",
    end="\n\n")

Top 3 results:
1:
    Page Content: - **August 2008 - May 2012:** Software Engineer at TechStartup
  - Developed features for SaaS platform
  - Gained experience across full technology stack

## Annual Performance History
- **2023:** Ra...
    Tag: employees
    Source: ../knowledge-base/employees/Robert Chen.md

2:
    Page Content: 1. **Unlimited Member Administration:** No capacity restrictions, supporting United's 250,000+ members with scalability to 1 million+ members as business expands.

2. **Multi-State Operations:** Compl...
    Tag: contracts
    Source: ../knowledge-base/contracts/Contract with United Healthcare Alliance for Healthllm.md

3:
    Page Content: # HR Record

# James Wilson

## Summary
- **Date of Birth:** April 5, 1978
- **Job Title:** Chief Technology Officer (CTO)
- **Location:** San Francisco, California
- **Current Salary:** $285,000

## ...
    Tag: employees
    Source: ../knowledge-base/employees/James Wilson.md



In [14]:
# Let's investigate the vectors
collection = vectordb._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 413 vectors with 384 dimensions in the vector store


Visualize

In [ ]:
# Prework
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
# Get the number of documents of each type (products, employees, contracts, company) in the vector store.
doc_type_counts = {}
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]

for t in doc_types:
    doc_type_counts[t] = doc_type_counts.get(t, 0) + 1
print("Document counts by type:")
for doc_type, count in doc_type_counts.items():
    print(f"{doc_type}: {count}")

colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

Document counts by type:
contracts: 230
company: 18
employees: 118
products: 47


In [ ]:
# We humans find it easier to visualize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)

from sklearn.manifold import TSNE
import plotly.graph_objects as go

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [24]:
# Lets use a VectorStoreRetriever to retrieve the relevant chunks from the vector database based on a query.
retriever = vectordb.as_retriever()
retriever.invoke("Daniel Park")

[Document(id='621a717b-26ef-4cd4-be14-11e92b58f2ba', metadata={'source': '../knowledge-base/contracts/Contract with Guardian Life Partners for Lifellm.md', 'doc_type': 'contracts'}, page_content="_________________________________\n**Jonathan Park**\n**Title**: President & CEO\n**Guardian Life Partners**\n**Date**: March 1, 2025\n\n---\n\n*This contract establishes Guardian Life Partners as a strategic partner leveraging Lifellm's advanced AI underwriting and digital health integration to modernize life insurance operations.*"),
 Document(id='b23a6f84-3cb1-4e65-b5ef-3858d850307a', metadata={'source': '../knowledge-base/employees/Jordan K. Bishop.md', 'doc_type': 'employees'}, page_content="## Other HR Notes\n- Jordan K. Bishop has been an integral part of club initiatives, including the Insurellm Code Reviews and Feedback Group, providing peer support.\n- Active participant in the company's Diversity and Inclusion committee, promoting a positive work culture.\n- Jordan has expressed int

In [28]:
from langchain_openai import ChatOpenAI
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

# Instantiate the OpenAI chat models from Langchain.
# llm = ChatOpenAI(model=MODEL, temperature=0)

llm = HuggingFaceEndpoint(
    model=MODEL,
    temperature=0,
    huggingfacehub_api_token=os.getenv("HF_TOKEN")
)

chat_llm = ChatHuggingFace(llm=llm)


In [17]:
SYSTEM_MESSAGE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [21]:
# Create an utility function to answer user queries based on the vector retriever.
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

def get_answer(query: str, history=None):
    history_langchain_format = []
    history = history or []
    for msg in history:
        if msg['role'] == 'user':
            history_langchain_format.append(HumanMessage(content=msg['content']))
        elif msg['role'] == 'assistant':
            history_langchain_format.append(AIMessage(content=msg['content']))
    relevant_docs = retriever.invoke(query)
    context = "\n\n".join(doc.page_content for doc in relevant_docs)
    system_prompt = SYSTEM_MESSAGE.format(context=context)
    messages = [
        SystemMessage(content=system_prompt),
        *history_langchain_format,
        HumanMessage(content=query)
    ]
    response = chat_llm.invoke(messages)
    return response.content

In [ ]:
# Test the utility function with a query.
query = "Name some of the employees of this company?"
answer = get_answer(query)
print(f"Answer: {answer}")

Answer: Here are a few of the employees listed in the HR records for Insurellm:

| Name | Job Title | Location |
|------|-----------|----------|
| **Robert Chen** | Senior Full Stack Engineer | San Francisco, CA |
| **Marcus Johnson** | Customer Success Manager | New York, NY |
| **James Wilson** | Chief Technology Officer (CTO) | San Francisco, CA |
| **Amanda Foster** | HR Business Partner | San Francisco, CA |

These are some of the current team members whose profiles are included in the provided records.


In [23]:
# Render a Gradio Chat app to have a conversation with the assistant.
import gradio as gr
demo = gr.ChatInterface(get_answer, type="messages")
demo.launch(server_name="0.0.0.0", server_port=7860)

* Running on local URL:  http://0.0.0.0:7860
* To create a public link, set `share=True` in `launch()`.


In [22]:
demo.close()

Closing server running on port: 7860
